In [ ]:
!pip3 install pandas matplotlib numpy seaborn

* Nota média por município
* Apenas a título de ilustração, imputar o missing na prova de Matemática pela média ou mediana de algum grupo ou grupos escolhido.
* Quais municípios possuem a maior quantidade de inscritos.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.set_option('display.max_columns', 100)

In [ ]:
df = pd.read_csv('dados_enem_2021_BA.csv')

In [ ]:
df.shape

In [ ]:
# 5 primeiras linhas
df.head()

* Média na nota de Matemática por gênero

In [ ]:
subset_tp_sexo_m = df.query('TP_SEXO == "M"')

In [ ]:
subset_tp_sexo_f = df.query('TP_SEXO == "F"')

Agregação:
* `df.column.agg_func()`

In [ ]:
# nota máximo do Gênero masculino
subset_tp_sexo_m.NU_NOTA_MT.max()

In [ ]:
# nota máxima do gênero feminino
subset_tp_sexo_f.NU_NOTA_MT.max()

In [ ]:
# nota média do Gênero masculino
subset_tp_sexo_m.NU_NOTA_MT.mean()

In [ ]:
# nota máxima do gênero feminino
subset_tp_sexo_f.NU_NOTA_MT.mean()

In [ ]:
# nota média do Gênero masculino
subset_tp_sexo_m.NU_NOTA_MT.median()

In [ ]:
# nota máxima do gênero feminino
subset_tp_sexo_f.NU_NOTA_MT.median()

In [ ]:
provas = df.columns[(df.columns.str.contains('NOTA')) & (~df.columns.str.contains('COMP'))].tolist()
idCandidato = ['NU_INSCRICAO']

In [ ]:
subset_tp_sexo_m[provas].agg([np.min, np.mean, np.median, np.max]).T.round(2)

In [ ]:
subset_tp_sexo_f[provas].agg([np.min, np.mean, np.median, np.max]).T.round(2)

* Qual foi o aluno que tirou a nota máxima/mínima?

In [ ]:
df.NU_NOTA_MT.max()

In [ ]:
df.NU_NOTA_MT.idxmax()

In [ ]:
df.iloc[df.NU_NOTA_MT.idxmax(),]

In [ ]:
df.iloc[df.NU_NOTA_MT.idxmin(),]

* Nota mínima na redação

O que representa o NaN em uma prova?

In [ ]:
df[df.NU_NOTA_REDACAO.notna()].NU_NOTA_REDACAO.isna().sum()

In [ ]:
mask_1 = df.NU_NOTA_REDACAO.notna()
mask_2 = df.NU_NOTA_REDACAO != 0
subset = df[(mask_1) & (mask_2)]

In [ ]:
df.NU_NOTA_REDACAO.plot(kind = 'hist')

In [ ]:
df.NU_NOTA_REDACAO.plot(kind = 'box')

In [ ]:
subset.NU_NOTA_REDACAO.plot(kind = 'hist')

In [ ]:
subset.NU_NOTA_REDACAO.plot(kind = 'box')

In [ ]:
subset.NU_NOTA_REDACAO.agg([np.min, np.mean, np.median, np.max])

* Axis na função de agregação

In [ ]:
df[provas].mean()

In [ ]:
# média das colunas
df[provas].mean(axis = 0)

In [ ]:
# média dos alunos
df[provas].mean(axis = 1)

In [ ]:
(df.NU_NOTA_CH+df.NU_NOTA_LC+df.NU_NOTA_MT+df.NU_NOTA_CN+df.NU_NOTA_REDACAO) / 5

* Missing

Imputar missing : `.fillna()`

In [ ]:
# missing simbólico
# -1
df_copy = df.copy()
df_copy['MEAN'] = df[provas].mean(axis = 1)

In [ ]:
df_copy.MEAN.fillna(-1)

Imputar pela média

In [ ]:
mean = np.mean(df_copy.MEAN)

In [ ]:
df_copy.MEAN.hist()

In [ ]:
df_copy.MEAN.fillna(mean).hist()

Imputar pela mediana

In [ ]:
df_copy.MEAN.median()

In [ ]:
median = df_copy.MEAN.median()

In [ ]:
df_copy.MEAN.fillna(median)

* Agrupamento

Qual a proporção entre os gêneros?

In [ ]:
df.TP_SEXO.value_counts()

In [ ]:
df.groupby(by = ['TP_SEXO'])['NU_INSCRICAO'].count()

In [ ]:
df.TP_ESCOLA.value_counts()

In [ ]:
df.groupby(by = ['TP_SEXO', 'TP_ESCOLA'])['NU_INSCRICAO'].count()

Qual a distribuição de frequência dos alunos por tipo de escola?

In [ ]:
df.groupby(by = ['TP_ESCOLA'])['NU_INSCRICAO'].count().sort_index()

Qual o desempenho em Matemática por tipo de escola?

In [ ]:
df.groupby(by = ['TP_ESCOLA'])['NU_NOTA_MT'].mean()

In [ ]:
df.groupby(by = ['TP_ESCOLA'])[['NU_NOTA_MT', 'NU_NOTA_CN']].mean()

In [ ]:
df.groupby(by = ['TP_ESCOLA'])[['NU_NOTA_MT', 'NU_NOTA_CN']].agg([np.min, np.median, np.mean, np.std, np.max]).T

In [ ]:
(
    df
    .groupby(by = ['TP_ESCOLA'])[['NU_NOTA_MT', 'NU_NOTA_CN']]
    .agg([np.min, np.median, np.mean, np.std, np.max])
    .T
)

In [ ]:
(
    df.groupby(by = ['TP_ESCOLA'])
    .agg({
        'NU_NOTA_MT' : [np.mean, np.median],
        'NU_NOTA_CN' : [np.min, np.max]
    }
    )
)

* Nota média por município

In [ ]:
df.head()

In [ ]:
df.NO_MUNICIPIO_PROVA

In [ ]:
df.NU_NOTA_MT.hist()

In [ ]:
df_visao_municipio = (
    df
    .query('NU_NOTA_MT != 0 ')
    .groupby(by = ['NO_MUNICIPIO_PROVA', 'CO_MUNICIPIO_PROVA'], as_index = False)['NU_NOTA_MT']
    .agg([np.min, np.mean, np.median, np.max])
    .reset_index(drop = False)
    .rename(
        columns = {
            'CO_MUNICIPIO_PROVA' : 'COD_IBGE',
            'NO_MUNICIPIO_PROVA': 'Município',
            'amin' : 'Mínimo_MT',
            'mean' : 'Média_MT',
            'median' : 'Mediana_MT',
            'amax' : 'Máximo_MT'
                  })
    .sort_values(by = ['Máximo_MT', 'Média_MT', 'Mediana_MT'], ascending = False)
    .reset_index(drop = True)
)

In [ ]:
df_visao_municipio.head()

In [ ]:
df_quantidade_inscritos = (
    df.groupby(by = ['NO_MUNICIPIO_PROVA', 'CO_MUNICIPIO_PROVA'], as_index = False)['NU_INSCRICAO']
    .count()
    .rename(columns = {'NO_MUNICIPIO_PROVA' : 'Município',
                       'CO_MUNICIPIO_PROVA' : 'COD_IBGE',
                       'NU_INSCRICAO' : 'Quantidade_inscritos'
                      })
    .sort_values(by = ['Quantidade_inscritos'], ascending = False)
    .reset_index(drop = True)
)

In [ ]:
total = df_quantidade_inscritos.Quantidade_inscritos.sum()

In [ ]:
df_quantidade_inscritos['Percentual_inscritos'] = (df_quantidade_inscritos.Quantidade_inscritos / total * 100).round(2)

In [ ]:
df_quantidade_inscritos['NU_ANO'] = 2021

In [ ]:
df_visao_municipio.head()

In [ ]:
df_quantidade_inscritos.head()

In [ ]:
df_municipio = pd.merge(
    df_visao_municipio.drop(columns = ['Município']),
    df_quantidade_inscritos,
    on = 'COD_IBGE',
    how = 'inner'
)

In [ ]:
df_municipio.head(10)

In [ ]:
df_visao_municipio.merge(df_quantidade_inscritos, how = 'inner')

Crosstab